In [ ]:

import pandas as pd
import numpy as np
import csv
import json
import requests

print("CARELOOP CLINICS - NO-SHOW GUARD")
print("Python environment ready!")
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)


CARELOOP CLINICS - NO-SHOW GUARD
Python environment ready!
Pandas version: 2.2.3
NumPy version: 2.4.6


In [ ]:

import pandas as pd

appointments = pd.read_csv("/content/appointments.csv")
clinics = pd.read_csv("/content/clinics.csv")
notes = pd.read_csv("/content/followup_notes.csv")

print("CARELOOP DATASET VERIFICATION")
print("-----------------------------")
print("Appointments:", appointments.shape)
print("Clinics:", clinics.shape)
print("Follow-up Notes:", notes.shape)
print("-----------------------------")
print("All datasets loaded successfully!")


CARELOOP DATASET VERIFICATION
-----------------------------
Appointments: (1020, 14)
Clinics: (8, 6)
Follow-up Notes: (30, 3)
-----------------------------
All datasets loaded successfully!


## Task 1.1 — Raw CSV Ingestion and Type Validation

**Objective:** Read the original appointment dataset using Python's built-in CSV module, validate numeric fields, separate successfully parsed records from rejected records, and export rejected records to a JSON file.

This step creates an audit trail of invalid records before the main Pandas cleaning process.


In [ ]:

import csv
import json

accepted = []
rejected = []

def parse_row(row):
    row["age"] = int(row["age"])
    row["fee_inr"] = int(row["fee_inr"])
    row["prior_no_shows"] = int(row["prior_no_shows"])
    row["distance_km"] = float(row["distance_km"])

    return row

with open("/content/appointments.csv", "r", newline="") as file:

    reader = csv.DictReader(file)

    print("Column names:")
    print(reader.fieldnames)

    for row in reader:
        try:
            accepted.append(parse_row(row))
        except ValueError:
            rejected.append(row)

with open("/content/rejected_rows.json", "w") as file:
    json.dump(rejected, file, indent=2)

print("\nRESULTS")
print("Total records:", len(accepted) + len(rejected))
print("Accepted records:", len(accepted))
print("Rejected records:", len(rejected))
print("JSON file created successfully!")


Column names:
['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']

RESULTS
Total records: 1020
Accepted records: 988
Rejected records: 32
JSON file created successfully!


## Task 1.2 — Data Inspection and Cleaning (Pass 1)

**Objective:** Inspect the raw appointment dataset to identify missing values, duplicate records and inconsistent booking-channel labels. Remove exact duplicates, standardize categorical labels, convert date columns, and remove records with invalid appointment dates or missing attendance status.


In [ ]:

# TASK 1.2 — DATA INSPECTION

import pandas as pd

# Read the ORIGINAL dataset
df = pd.read_csv("/content/appointments.csv")

print("1. ORIGINAL DATASET SHAPE")
print(df.shape)

print("\n2. DATASET INFORMATION")
df.info()

print("\n3. MISSING VALUES")
print(df.isnull().sum())

print("\n4. DUPLICATE RECORDS")
print(df.duplicated().sum())

print("\n5. RAW BOOKING CHANNEL VALUES")
print(df["booking_channel"].value_counts(dropna=False))


1. ORIGINAL DATASET SHAPE
(1020, 14)

2. DATASET INFORMATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

3. MISSING VALUES
appointment_

In [ ]:

# TASK 1.2 - DATA CLEANING

# A. Remove exact duplicates
df = df.drop_duplicates().copy()

print("After removing duplicates:", len(df))

# B. Standardize booking channels
df["booking_channel"] = (
    df["booking_channel"]
    .str.strip()
    .str.title()
)

print("\nBooking channels after cleaning:")
print(df["booking_channel"].value_counts())

# C. Convert dates
df["appt_date"] = pd.to_datetime(
    df["appt_date"],
    errors="coerce"
)

df["booked_on"] = pd.to_datetime(
    df["booked_on"],
    errors="coerce"
)

# D. Remove records with invalid appointment
# dates or missing attendance status
df = df.dropna(
    subset=["appt_date", "status"]
).copy()

print("\nFinal records:", len(df))
print("Final dataset shape:", df.shape)


After removing duplicates: 1000

Booking channels after cleaning:
booking_channel
App        387
Phone      359
Walk-In    254
Name: count, dtype: int64

Final records: 990
Final dataset shape: (990, 14)


/tmp/ipykernel_3976/1361768532.py:19: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["appt_date"] = pd.to_datetime(
/tmp/ipykernel_3976/1361768532.py:24: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["booked_on"] = pd.to_datetime(


In [2]:
import pandas as pd
df["appt_date"] = pd.to_datetime(
    df["appt_date"],
    errors="coerce",
    dayfirst=True
)

df["booked_on"] = pd.to_datetime(
    df["booked_on"],
    errors="coerce",
    dayfirst=True
)


NameError: name 'df' is not defined

In [5]:

# CARELOOP CLINICS - TASK 1.2
# Reload and clean the original dataset

import pandas as pd

# Load original appointment data
df = pd.read_csv("/content/appointments.csv")

print("Original records:", len(df))

# Remove exact duplicates
df = df.drop_duplicates().copy()

print("After removing duplicates:", len(df))

# Standardize booking channels
df["booking_channel"] = (
    df["booking_channel"]
    .str.strip()
    .str.title()
)

# Convert appointment dates
df["appt_date"] = pd.to_datetime(
    df["appt_date"],
    errors="coerce",
    dayfirst=True
)

# Convert booking dates
df["booked_on"] = pd.to_datetime(
    df["booked_on"],
    errors="coerce",
    dayfirst=True
)

# Remove missing appointment dates and status
df = df.dropna(
    subset=["appt_date", "status"]
).copy()

print("Final records:", len(df))
print("Final shape:", df.shape)

print("\nBooking channels:")
print(df["booking_channel"].value_counts())


FileNotFoundError: [Errno 2] No such file or directory: '/content/appointments.csv'

In [7]:

import os

print("Files available in Colab:")
print(os.listdir("/content"))



Files available in Colab:
['.config', 'sample_data']


In [8]:

import os
import shutil

files = [
    "appointments.csv",
    "clinics.csv",
    "followup_notes.csv"
]

for filename in files:
    source = "/" + filename
    destination = "/content/" + filename

    if os.path.exists(source):
        shutil.copy2(source, destination)
        print("Copied:", filename)
    elif os.path.exists(destination):
        print("Already available:", filename)
    else:
        print("Not found:", filename)

print("\nFiles in /content:")
print(os.listdir("/content"))


Copied: appointments.csv
Copied: clinics.csv
Copied: followup_notes.csv

Files in /content:
['.config', 'followup_notes.csv', 'clinics.csv', 'appointments.csv', 'sample_data']


In [9]:

# ==========================================
# CARELOOP CLINICS - NO-SHOW GUARD
# TASK 1.2 - DATA INSPECTION AND CLEANING
# ==========================================

import pandas as pd

# STEP 1: Load the original dataset
df = pd.read_csv("/content/appointments.csv")

print("Original records:", len(df))
print("Original shape:", df.shape)

# STEP 2: Inspect the dataset
print("\nMissing values before cleaning:")
print(df.isnull().sum())

print("\nExact duplicate records:")
print(df.duplicated().sum())

print("\nRaw booking channel spellings:")
print(df["booking_channel"].nunique())

# STEP 3: Remove exact duplicate records
df = df.drop_duplicates().copy()

print("\nAfter removing duplicates:", len(df))

# STEP 4: Standardize booking channel values
df["booking_channel"] = (
    df["booking_channel"]
    .str.strip()
    .str.title()
)

print("\nBooking channels after cleaning:")
print(df["booking_channel"].value_counts())

# STEP 5: Convert date columns
df["appt_date"] = pd.to_datetime(
    df["appt_date"],
    errors="coerce",
    dayfirst=True
)

df["booked_on"] = pd.to_datetime(
    df["booked_on"],
    errors="coerce",
    dayfirst=True
)

# STEP 6: Remove missing appointment dates
# and missing attendance status
df = df.dropna(
    subset=["appt_date", "status"]
).copy()

# STEP 7: Verify the cleaned dataset
print("\n========== FINAL RESULTS ==========")
print("Final records:", len(df))
print("Final dataset shape:", df.shape)

print("\nMissing appointment dates:",
      df["appt_date"].isna().sum())

print("Missing attendance status:",
      df["status"].isna().sum())

print("\nFinal booking channels:")
print(df["booking_channel"].unique())

print("\nTASK 1.2 COMPLETED!")


Original records: 1020
Original shape: (1020, 14)

Missing values before cleaning:
appointment_id      0
patient_id          0
clinic_id           0
booked_on           0
appt_date           1
age                12
gender              0
distance_km        20
fee_inr             0
booking_channel     0
visit_type          0
reminder_sent       0
prior_no_shows      0
status              4
dtype: int64

Exact duplicate records:
20

Raw booking channel spellings:
16

After removing duplicates: 1000

Booking channels after cleaning:
booking_channel
App        387
Phone      359
Walk-In    254
Name: count, dtype: int64

========== FINAL RESULTS ==========
Final records: 990
Final dataset shape: (990, 14)

Missing appointment dates: 0
Missing attendance status: 0

Final booking channels:
['Phone' 'Walk-In' 'App']

TASK 1.2 COMPLETED!


## Task 1.3 — Data Cleaning: Validity, Missing Values and Outliers

**Objective:** Improve dataset quality by correcting invalid ages, handling missing numerical data, and capping extreme values using the Interquartile Range (IQR) method.

**Methodology:**
- Treat ages below 1 or above 100 as invalid.
- Replace invalid ages with missing values.
- Fill missing ages and distances using their medians.
- Calculate Q1, Q3 and IQR.
- Cap extreme distance and consultation fee values.
- Verify the number of corrected records.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [3]:

# ==========================================
# CARELOOP CLINICS - NO-SHOW GUARD
# PART 2 - PYTHON VS SQL VERIFICATION
# ==========================================

import pandas as pd
import os

# STEP 1: Check required files

print("CHECKING REQUIRED FILES")

appointment_path = "/content/appointments_clean.csv"
clinic_path = "/content/clinics_enriched.csv"

print("Appointments file:", os.path.exists(appointment_path))
print("Clinics file:", os.path.exists(clinic_path))

assert os.path.exists(appointment_path), "Upload appointments_clean.csv"
assert os.path.exists(clinic_path), "Upload clinics_enriched.csv"

# STEP 2: Load cleaned datasets

appointments = pd.read_csv(appointment_path)
clinics = pd.read_csv(clinic_path)

print("\nDATASET VERIFICATION")
print("Total appointments:", len(appointments))
print("Total clinics:", len(clinics))
print("Total no-shows:", appointments["no_show"].sum())

# STEP 3: Group appointments by clinic

summary = (
    appointments.groupby("clinic_id")
    .agg(
        appointments=("appointment_id", "count"),
        no_shows=("no_show", "sum")
    )
    .reset_index()
)

# STEP 4: Merge clinic details

summary = summary.merge(
    clinics[["clinic_id", "clinic_name", "city"]],
    on="clinic_id",
    validate="one_to_one"
)

# STEP 5: Calculate no-show percentage

summary["no_show_pct"] = (
    100 * summary["no_shows"] / summary["appointments"]
).round(1)

# STEP 6: Apply SQL HAVING condition

summary = summary[
    summary["appointments"] >= 95
].sort_values(
    ["no_show_pct", "clinic_name"],
    ascending=[False, True]
)

# STEP 7: Display results

print("\nPYTHON CLINIC-WISE ANALYSIS")

print(
    summary[
        [
            "clinic_name",
            "city",
            "appointments",
            "no_shows",
            "no_show_pct"
        ]
    ].to_string(index=False)
)

# STEP 8: Validate results

assert len(appointments) == 990
assert len(clinics) == 8
assert appointments["no_show"].sum() == 218

assert len(summary) == 6
assert summary["appointments"].sum() == 818
assert summary["no_shows"].sum() == 175

print("\n===================================")
print("PYTHON VS SQL VERIFICATION PASSED")
print("All reference counts match!")
print("===================================")


CHECKING REQUIRED FILES
Appointments file: True
Clinics file: True

DATASET VERIFICATION
Total appointments: 990
Total clinics: 8
Total no-shows: 218

PYTHON CLINIC-WISE ANALYSIS
             clinic_name   city  appointments  no_shows  no_show_pct
    Hadapsar Care Centre   Pune           143        39         27.3
    Malviya Nagar Clinic Jaipur           142        38         26.8
   Kothrud Family Clinic   Pune           160        36         22.5
      Vijay Nagar Clinic Indore           101        21         20.8
Edappally Medical Centre  Kochi           142        23         16.2
   Vaishali Nagar Clinic Jaipur           130        18         13.8

PYTHON VS SQL VERIFICATION PASSED
All reference counts match!


In [10]:

# ==========================================
# CARELOOP CLINICS - NO-SHOW GUARD
# TASK 1.3 - INVALID AGES AND MISSING VALUES
# ==========================================

import numpy as np

# STEP 1: Identify invalid ages
invalid_age = (df["age"] < 1) | (df["age"] > 100)

print("Invalid ages found:", invalid_age.sum())

# STEP 2: Replace invalid ages with NaN
df.loc[invalid_age, "age"] = np.nan

# STEP 3: Calculate medians
age_median = df["age"].median()
distance_median = df["distance_km"].median()

print("Median age:", age_median)
print("Median distance:", distance_median)

# STEP 4: Fill missing ages
df["age"] = (
    df["age"]
    .fillna(age_median)
    .round()
    .astype(int)
)

# STEP 5: Fill missing distances
df["distance_km"] = df["distance_km"].fillna(
    distance_median
)

# STEP 6: Verify missing values
print("\nMissing values after filling:")
print(df[["age", "distance_km"]].isnull().sum())


Invalid ages found: 5
Median age: 48.0
Median distance: 8.25

Missing values after filling:
age            0
distance_km    0
dtype: int64


In [11]:

# TASK 1.3 - IQR OUTLIER TREATMENT

for column in ["distance_km", "fee_inr"]:

    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)

    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    capped_count = (
        (df[column] < lower) |
        (df[column] > upper)
    ).sum()

    df[column] = df[column].clip(
        lower=lower,
        upper=upper
    )

    if column == "distance_km":
        df[column] = df[column].round(1)
    else:
        df[column] = df[column].round().astype(int)

    print("\nColumn:", column)
    print("Q1:", q1)
    print("Q3:", q3)
    print("IQR:", iqr)
    print("Upper fence:", upper)
    print("Values capped:", capped_count)



Column: distance_km
Q1: 4.625
Q3: 13.0
IQR: 8.375
Upper fence: 25.5625
Values capped: 26

Column: fee_inr
Q1: 500.0
Q3: 787.5
IQR: 287.5
Upper fence: 1218.75
Values capped: 3


## Task 1.4 — Feature Engineering and NumPy Statistical Analysis

**Objective:** Create new analytical variables to understand patient appointment behavior and prepare the dataset for machine learning.

**Methodology:**
- Calculate booking lead time (`lead_days`).
- Create the binary target variable (`no_show`).
- Use NumPy to calculate feature averages.
- Identify appointments booked more than 14 days in advance.
- Calculate standardized distance values using z-scores.
- Summarize important appointment statistics.

In [12]:

# ==========================================
# CARELOOP CLINICS - NO-SHOW GUARD
# TASK 1.4 - FEATURE ENGINEERING & NUMPY
# ==========================================

import numpy as np

# STEP 1: Calculate booking lead time
df["lead_days"] = (
    df["appt_date"] - df["booked_on"]
).dt.days

# STEP 2: Create the binary target variable
df["no_show"] = (
    df["status"] == "No-Show"
).astype(int)

# STEP 3: Convert selected columns to NumPy
X = df[
    ["age", "distance_km", "fee_inr", "lead_days"]
].to_numpy()

# STEP 4: Calculate column means without loops
means = np.mean(X, axis=0)

print("Column means:", means)

# STEP 5: Calculate bookings more than 14 days ahead
mask = df["lead_days"] > 14

print("\nBookings with lead_days > 14:", mask.sum())
print(
    "Percentage:",
    round(mask.mean() * 100, 1)
)

# STEP 6: Calculate distance z-scores
distance = df["distance_km"].to_numpy()

z = (
    distance - np.mean(distance)
) / np.std(distance)

print(
    "\nRows with |z| > 2:",
    np.sum(np.abs(z) > 2)
)

# STEP 7: Summarize no-shows
print("\nTotal appointments:", len(df))
print("Total no-shows:", df["no_show"].sum())
print(
    "No-show percentage:",
    round(df["no_show"].mean() * 100, 1)
)

print("\nLead days minimum:", df["lead_days"].min())
print("Lead days maximum:", df["lead_days"].max())

print("\nTASK 1.4 COMPLETED!")


Column means: [ 48.17373737   9.50090909 635.71414141  11.81919192]

Bookings with lead_days > 14: 296
Percentage: 29.9

Rows with |z| > 2: 53

Total appointments: 990
Total no-shows: 218
No-show percentage: 22.0

Lead days minimum: 1
Lead days maximum: 45

TASK 1.4 COMPLETED!


## Task 1.5 — Clinic Data Enrichment Using Open-Meteo API

**Objective:** Integrate an external API with the CareLoop Clinics dataset to enrich clinic records with temperature information.

**Methodology:**
- Load the eight clinic records.
- Use each clinic's latitude and longitude to request hourly temperature data.
- Retrieve the first temperature reading from the API response.
- Handle failed requests using exception handling and one retry.
- Store missing temperature values when the API fails.
- Export the enriched dataset as `clinics_enriched.csv`.

**Note:** The API reading is used as a demonstration of data enrichment, not as historical weather on the appointment dates.

In [13]:

# ==========================================
# CARELOOP CLINICS - NO-SHOW GUARD
# TASK 1.5 - LIVE API DATA ENRICHMENT
# ==========================================

import pandas as pd
import requests

# Load clinic data
clinics = pd.read_csv("/content/clinics.csv")

print("Total clinics:", len(clinics))


def get_temperature(latitude, longitude, clinic_name):

    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": "temperature_2m",
        "forecast_days": 1
    }

    # Initial request plus one retry
    for attempt in range(2):

        try:
            response = requests.get(
                url,
                params=params,
                timeout=20
            )

            if response.status_code != 200:
                raise ValueError(
                    f"HTTP status {response.status_code}"
                )

            data = response.json()

            temperature = data["hourly"]["temperature_2m"][0]

            if temperature is None:
                raise ValueError("Temperature missing")

            return temperature

        except (
            requests.exceptions.RequestException,
            KeyError,
            IndexError,
            TypeError,
            ValueError
        ) as error:

            print(
                f"{clinic_name}: attempt {attempt + 1} failed - {error}"
            )

    # Return missing value if both attempts fail
    print(f"{clinic_name}: storing missing temperature")
    return None


# Collect temperatures for all eight clinics
temperatures = []

for _, clinic in clinics.iterrows():

    temperature = get_temperature(
        clinic["latitude"],
        clinic["longitude"],
        clinic["clinic_name"]
    )

    temperatures.append(temperature)

    print(
        clinic["clinic_name"],
        "- Temperature:",
        temperature
    )


# Add new temperature column
clinics["temp_c"] = temperatures

# Save enriched dataset
clinics.to_csv(
    "/content/clinics_enriched.csv",
    index=False
)

print("\n========== FINAL RESULTS ==========")
print("Total clinic records:", len(clinics))
print("Temperature values available:",
      clinics["temp_c"].notna().sum())

print("\nEnriched clinic data:")
print(clinics)

print("\nclinics_enriched.csv created successfully!")


Total clinics: 8
Kothrud Family Clinic - Temperature: 17.3
Hadapsar Care Centre - Temperature: 17.4
Sitabuldi Health Point - Temperature: 20.4
Malviya Nagar Clinic - Temperature: 20.7
Vaishali Nagar Clinic - Temperature: 20.7
Edappally Medical Centre - Temperature: 25.4
Vijay Nagar Clinic - Temperature: 20.4
Gomti Nagar Care Point - Temperature: 19.3

========== FINAL RESULTS ==========
Total clinic records: 8
Temperature values available: 8

Enriched clinic data:
  clinic_id               clinic_name     city    tier  latitude  longitude  \
0       C01     Kothrud Family Clinic     Pune   Metro   18.5074    73.8077   
1       C02      Hadapsar Care Centre     Pune   Metro   18.5089    73.9260   
2       C03    Sitabuldi Health Point   Nagpur  Tier-2   21.1458    79.0882   
3       C04      Malviya Nagar Clinic   Jaipur  Tier-2   26.8553    75.8150   
4       C05     Vaishali Nagar Clinic   Jaipur  Tier-2   26.9124    75.7873   
5       C06  Edappally Medical Centre    Kochi  Tier-2   

## Task 1.6 — Export Cleaned Data and Generate SQL Load Script

**Objective:** Prepare the final cleaned appointment dataset and generate SQL INSERT statements for importing the appointment and clinic records into a relational database.

**Methodology:**
- Sort appointment records by appointment ID.
- Format booking dates and appointment dates as YYYY-MM-DD.
- Organize the final dataset into 16 required columns.
- Check that appointment IDs are unique.
- Verify that no missing values remain.
- Export the cleaned appointment dataset as CSV.
- Generate SQL INSERT statements for clinics and appointments.
- Ensure that clinic records are inserted before appointments to maintain foreign-key relationships.

In [16]:

# ==========================================
# CARELOOP CLINICS - NO-SHOW GUARD
# TASK 1.6 - EXPORT CLEANED DATA
# ==========================================

import pandas as pd

# STEP 1: Sort appointments by ID
df = df.sort_values(
    "appointment_id"
).reset_index(drop=True)

# STEP 2: Ensure both date columns are datetime
for col in ["booked_on", "appt_date"]:
    df[col] = pd.to_datetime(
        df[col],
        errors="coerce",
        dayfirst=True
    )

# STEP 3: Format dates as YYYY-MM-DD
df["booked_on"] = df["booked_on"].dt.strftime("%Y-%m-%d")
df["appt_date"] = df["appt_date"].dt.strftime("%Y-%m-%d")

# STEP 4: Define required column order
columns = [
    "appointment_id",
    "patient_id",
    "clinic_id",
    "booked_on",
    "appt_date",
    "age",
    "gender",
    "distance_km",
    "fee_inr",
    "booking_channel",
    "visit_type",
    "reminder_sent",
    "prior_no_shows",
    "status",
    "lead_days",
    "no_show"
]

# STEP 5: Arrange columns
df = df[columns]

# STEP 6: Verify the dataset
print("Total rows:", len(df))
print("Total columns:", len(df.columns))
print("Unique appointment IDs:",
      df["appointment_id"].nunique())
print("Missing values:", df.isna().sum().sum())

# STEP 7: Validate
assert df.shape == (990, 16)
assert df["appointment_id"].is_unique
assert not df.isna().any().any()

# STEP 8: Save cleaned CSV
df.to_csv(
    "/content/appointments_clean.csv",
    index=False
)

print("\nappointments_clean.csv created successfully!")
print(df.head())


Total rows: 990
Total columns: 16
Unique appointment IDs: 990
Missing values: 0

appointments_clean.csv created successfully!
  appointment_id patient_id clinic_id   booked_on   appt_date  age gender  \
0          A0001       P509       C05  2025-01-24  2025-01-25   60      F   
1          A0002       P046       C07  2025-05-27  2025-06-06   39      M   
2          A0003       P009       C02  2025-03-24  2025-03-30   63      F   
3          A0004       P482       C01  2025-04-18  2025-04-22   52      F   
4          A0005       P251       C01  2024-11-23  2025-01-02   64      F   

   distance_km  fee_inr booking_channel    visit_type reminder_sent  \
0         18.0      850           Phone  Consultation           Yes   
1         12.5      350         Walk-In     Follow-up            No   
2          4.1      850         Walk-In  Consultation            No   
3          7.5      750             App  Consultation           Yes   
4         19.0      700           Phone  Consultation   

/tmp/ipykernel_794/1936244380.py:15: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df[col] = pd.to_datetime(
/tmp/ipykernel_794/1936244380.py:15: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df[col] = pd.to_datetime(


In [17]:

# ==========================================
# CARELOOP CLINICS - NO-SHOW GUARD
# TASK 1.6 - GENERATE SQL LOAD SCRIPT
# ==========================================

import pandas as pd

def sql_value(value):
    if pd.isna(value):
        return "NULL"

    if isinstance(value, str):
        return "'" + value.replace("'", "''") + "'"

    return str(value)


# Clinic columns in SQL table order
clinic_columns = [
    "clinic_id",
    "clinic_name",
    "city",
    "tier",
    "latitude",
    "longitude",
    "temp_c"
]

# Read the two verified output files
clinics_sql = pd.read_csv(
    "/content/clinics_enriched.csv"
)

appointments_sql = pd.read_csv(
    "/content/appointments_clean.csv"
)

# Write SQL inserts
with open("/content/load_data.sql", "w") as file:

    # Insert clinics first
    for _, row in clinics_sql[clinic_columns].iterrows():

        values = ", ".join(
            sql_value(value) for value in row
        )

        file.write(
            f"INSERT INTO clinics VALUES ({values});\n"
        )

    # Insert appointments next
    for _, row in appointments_sql.iterrows():

        values = ", ".join(
            sql_value(value) for value in row
        )

        file.write(
            f"INSERT INTO appointments VALUES ({values});\n"
        )


# Verify SQL file
with open("/content/load_data.sql", "r") as file:
    statements = file.readlines()

clinic_count = sum(
    line.startswith("INSERT INTO clinics")
    for line in statements
)

appointment_count = sum(
    line.startswith("INSERT INTO appointments")
    for line in statements
)

print("Clinic INSERT statements:", clinic_count)
print("Appointment INSERT statements:", appointment_count)
print("Total INSERT statements:", len(statements))

assert clinic_count == 8
assert appointment_count == 990
assert len(statements) == 998

print("\nload_data.sql created successfully!")


Clinic INSERT statements: 8
Appointment INSERT statements: 990
Total INSERT statements: 998

load_data.sql created successfully!


In [18]:

# ==========================================
# CARELOOP CLINICS - NO-SHOW GUARD
# RESTORE TASK 1.1 - REJECTED ROWS JSON
# ==========================================

import csv
import json
import os

accepted = []
rejected = []

def parse_row(row):
    row["age"] = int(row["age"])
    row["fee_inr"] = int(row["fee_inr"])
    row["prior_no_shows"] = int(row["prior_no_shows"])
    row["distance_km"] = float(row["distance_km"])
    return row

# Read the ORIGINAL appointments CSV
with open(
    "/content/appointments.csv",
    "r",
    newline="",
    encoding="utf-8-sig"
) as file:

    reader = csv.DictReader(file)

    for row in reader:
        try:
            accepted.append(parse_row(row))
        except ValueError:
            rejected.append(row)

# Save rejected records
output_path = "/content/rejected_rows.json"

with open(output_path, "w", encoding="utf-8") as file:
    json.dump(rejected, file, indent=2)

# Verify results
print("Total records:", len(accepted) + len(rejected))
print("Accepted records:", len(accepted))
print("Rejected records:", len(rejected))
print("File exists:", os.path.exists(output_path))

assert len(accepted) == 988
assert len(rejected) == 32

print("\nrejected_rows.json restored successfully!")


Total records: 1020
Accepted records: 988
Rejected records: 32
File exists: True

rejected_rows.json restored successfully!


In [1]:

# PART 2 - PYTHON VS SQL VERIFICATION

import pandas as pd

appointments = pd.read_csv(
    "/content/appointments_clean.csv"
)

clinics = pd.read_csv(
    "/content/clinics_enriched.csv"
)

summary = (
    appointments.groupby("clinic_id")
    .agg(
        appointments=("appointment_id", "count"),
        no_shows=("no_show", "sum")
    )
    .reset_index()
)

summary = summary.merge(
    clinics[["clinic_id", "clinic_name", "city"]],
    on="clinic_id"
)

summary["no_show_pct"] = (
    100 * summary["no_shows"] / summary["appointments"]
).round(1)

summary = summary[
    summary["appointments"] >= 95
].sort_values("no_show_pct", ascending=False)

print(summary[
    ["clinic_name", "city", "appointments",
     "no_shows", "no_show_pct"]
].to_string(index=False))

assert len(summary) == 6
assert summary["appointments"].sum() == 818
assert summary["no_shows"].sum() == 175

print("\nPython clinic counts match the SQL reference!")


FileNotFoundError: [Errno 2] No such file or directory: '/content/appointments_clean.csv'

In [2]:

import os

print(
    "Appointments file:",
    os.path.exists("/content/appointments_clean.csv")
)

print(
    "Clinics file:",
    os.path.exists("/content/clinics_enriched.csv")
)


Appointments file: False
Clinics file: False
